# 부록 · 오류 찾기: 오류 메시지 없이 돌아가는 AI 코드의 오류를 검사로 잡기

이 노트북은 구글 **Colab**에서 바로 실행됩니다. 설명 글에는 실행할 것이 없고, 왼쪽에 `[ ]` 와 ▶ 가 붙은 상자만 코드입니다. **1 준비**부터 번호 차례대로 위에서 아래로 내려갑니다.

📖 본문 학습 페이지: [2장 · AI에게 시키고 검증하기](https://grow.minds.kr/textbooks/css-methods/causal/book/ch02-도구와-자료-여정과-의미.html)

오류 메시지 없이 돌아가는 오류 여섯 가지를 하나씩 잡습니다. 각 문제는 이렇게 생겼습니다.

1. **의도**(말로 적은 분석)를 읽고, **명세**(파일 · 열 · 집단 · 필터 · 기대 인원)를 먼저 정합니다.
2. **AI가 준 코드**는 한 줄씩 읽지 않습니다. 대신 그 코드에 걸 **검사**를 고릅니다.
3. **실행 셀**을 돌리면 버그판과 정답판의 숫자, 그리고 검사의 결과가 나옵니다. 검사가 실패한 것을 본 뒤에, 코드에서 어긋난 곳을 찾습니다.

> 왜 이 노트북인가. AI가 엉뚱한 열이나 엉뚱한 집단을 계산하면, 그 결과는 그럴듯한 숫자라 수 하나만 봐서는 안 걸린다. 그런 오류를 잡는 길은 코드를 눈으로 훑는 것이 아니라, 결과보다 먼저 정한 명세에 결과를 거는 검사다(2장 §2.4 · 부록 B). 이 노트북은 여섯 오류마다 어느 검사가 잡는지를 보인다.

## 1. 준비

아래 상자 **둘**을 차례로 실행하세요. 둘 다 해야 그다음이 돌아갑니다.

1. 첫째 = 자료와 코드를 내려받습니다. 몇 초 걸리고 「경고」 문구가 떠도 정상입니다. 상자 아래에 `/content/css-methods-causal-code` 가 찍히면 성공입니다.
2. 둘째 = 도구를 불러옵니다. **준비 끝**이라는 줄이 찍히면 됩니다.

⚠ 한참 뒤에 돌아와 `pd`·`exp`가 없다거나 파일이 없다는 오류가 나면 코랩이 연결을 끊은 것입니다. 자료는 내 컴퓨터가 아니라 코랩의 컴퓨터에 있었으므로 함께 사라졌습니다. **이 두 상자를 위에서부터 다시** 누르면 됩니다(첫째 상자는 여러 번 눌러도 됩니다).

> **어느 것이 상자인가.** 왼쪽에 `[ ]` 표시와 **▶** 단추가 붙은 것이 코드 상자입니다. 글씨에 색이 섞여 있습니다. 지금 읽고 있는 이런 글에는 `[ ]` 도 ▶ 도 없습니다.

In [ ]:
# 이 책의 데이터·코드를 코랩으로 내려받습니다(수 초). 다시 눌러도 됩니다.
%cd /content
![ -d css-methods-causal-code ] || git clone -q https://github.com/dataminds/css-methods-causal-code.git
%cd /content/css-methods-causal-code

In [ ]:
import pandas as pd, numpy as np
from scipy import stats

def load(name, clean=True):
    df = pd.read_csv(f"data/journey_{name}.csv")
    return df[df.attn_1 == 1] if clean and "attn_1" in df else df

def ols(y, X):                      # 절편 포함 최소제곱 → (계수, 표준오차, p, R^2)
    y = np.asarray(y, float)
    X1 = np.column_stack([np.ones(len(y))] + [np.asarray(x, float) for x in X])
    b, *_ = np.linalg.lstsq(X1, y, rcond=None)
    resid = y - X1 @ b
    n, k = X1.shape
    se = np.sqrt(np.diag(resid @ resid / (n - k) * np.linalg.inv(X1.T @ X1)))
    p = 2 * stats.t.sf(np.abs(b / se), n - k)
    r2 = 1 - (resid @ resid) / ((y - y.mean()) @ (y - y.mean()))
    return b, se, p, r2

def cohen_d(a, b):
    sp = np.sqrt(((len(a)-1)*a.std(ddof=1)**2 + (len(b)-1)*b.std(ddof=1)**2) / (len(a)+len(b)-2))
    return (a.mean() - b.mean()) / sp

def cronbach(items):
    items = np.asarray(items, float); k = items.shape[1]
    return k/(k-1) * (1 - items.var(axis=0, ddof=1).sum() / items.sum(axis=1).var(ddof=1))

print("준비 끝. 데이터와 도우미 함수를 불러왔습니다.")


💡 **방금 둘째 상자에 코드가 길게 펼쳐졌는데, 지금 이해하지 않아도 됩니다.**

이 책 전체가 쓰는 **도우미 넷**을 미리 만들어 두는 곳이라 그렇습니다. 지금은 이름과 하는 일만 훑고 넘어가세요. 안에 든 식은 필요한 장에서 하나씩 만납니다.

| 이름 | 하는 일 | 처음 쓰는 곳 |
|---|---|---|
| `load` | 자료 한 벌을 불러오고 주의 점검 실패자를 걸러 낸다 | 2장 |
| `cronbach` | 척도 신뢰도(알파) | 4장 |
| `cohen_d` | 두 집단 차이의 효과크기 | 13장 |
| `ols` | 회귀의 계수·표준오차·p·R² | 14장 |

⚠ `np.linalg.lstsq` 처럼 낯선 이름이 보여도 괜찮습니다. **그것이 이 책이 「계산은 AI가」라고 말하는 곳입니다.**

⭐ **그리고 이 넷은 여러분이 검증할 대상이 아닙니다.** 이 책이 검증해 둔 도구입니다(배포 때마다 대조 배터리가 이 함수들의 출력을 본문 수치와 맞춰 봅니다). 여러분이 읽어야 할 코드는 **AI가 준 코드**이고, 그 훈련은 3장부터 시작합니다.

📖 이 넷을 다시 보고 싶으면 책 부록 E.1b 에 같은 코드가 있습니다.

## 여섯 오류와 그 오류를 잡는 검사
이 노트북의 여섯 문제가 이 여섯 가지입니다. 넷째 칸의 검사는 코드를 읽지 않고 돌릴 수 있습니다.

| # | 오류 | 오류 메시지 없이 생기는 흔한 꼴 | 잡는 검사 (부록 B) |
|---|---|---|---|
| 1 | **집단** | 통제군 대신 전체와 비교, 엉뚱한 집단 필터 | 인원 검사 (집단마다 기대 인원) · 판 대조 |
| 2 | **열** | 결과변수 대신 기저·다른 척도 열 | 판 대조 (다른 AI의 판과 값이 어긋난다) |
| 3 | **표본·결측** | 정제 필터 누락, 다른 사람들의 평균끼리 비교 | 인원 검사 · 모의 세계 시험 |
| 4 | **경계** | `<` 와 `<=`, 중앙값 분할에서 사람 소실 | 인원 검사 (나눈 집단의 합 = 전체) |
| 5 | **방향** | 빼는 순서가 뒤집혀 결론이 정반대 | 극단값 시험 (한쪽에 값을 더하면 차이가 어느 쪽으로 가야 하나) |
| 6 | **기본값** | 표준편차 ddof, 양측·단측, 표준화 여부 | 판 대조 (언어 축 = 파이썬 대 R의 기본값) |

### 문제 1 · 집단

**의도**: 주의 점검을 통과한 사람(attn_1이 1)만으로, 개입 집단(cond==1)과 **통제 집단(cond==0)** 의 삶의 의미(mil) 평균 차이를 구한다.

AI가 준 코드:

```python
exp = exp[exp.attn_1 == 1]
gap = exp[exp.cond==1].mil.mean() - exp.mil.mean()   # 개입 - ???
```

**명세**: 정제본 366명 · 개입 183명 대 통제 183명. **잡는 검사**: 인원 검사 = 두 항이 각각 몇 명의 평균인가.

검사가 실패하면 코드에서 볼 곳: 둘째 항 `exp.mil.mean()` 은 무엇의 평균인가? 의도의 '통제 집단'과 같은가?

In [ ]:
exp = load("exp")
tg, cg = exp[exp.cond==1].mil, exp[exp.cond==0].mil
buggy = tg.mean() - exp.mil.mean()     # 개입 - 전체(개입 포함) 평균
right = tg.mean() - cg.mean()          # 개입 - 통제
print(f"버그(개입-전체): {buggy:+.3f}     바름(개입-통제): {right:+.3f}")
print(f"인원 검사: 둘째 항의 인원 {len(exp)}명 (명세 = 통제 집단 {len(cg)}명) → 어긋남")
print("→ 둘째 항이 통제군(cg)이 아니라 전체평균. 개입이 섞인 평균과 비교해 효과가 절반으로 줄어 보인다.")

### 문제 2 · 열

**의도**: 개입이 개입 **후** 삶의 의미(**mil**)를 높였는지 두 집단으로 비교한다.

AI가 준 코드:

```python
diff = exp[exp.cond==1].mil_t1.mean() - exp[exp.cond==0].mil_t1.mean()
```

**명세**: 결과 = 개입 뒤 삶의 의미(mil). **잡는 검사**: 판 대조 = 같은 부탁을 다른 AI에게 따로 시킨 판과 값이 맞는가.

검사가 실패하면 코드에서 볼 곳: `mil_t1` 은 어느 시점의 값인가? 의도의 결과변수와 같은 열인가?

In [ ]:
exp = load("exp")
for col in ["mil_t1", "mil"]:
    d = exp[exp.cond==1][col].mean() - exp[exp.cond==0][col].mean()
    print(f"{col:7s} 집단차 = {d:+.3f}")
gaps = {c: exp[exp.cond==1][c].mean() - exp[exp.cond==0][c].mean() for c in ["mil_t1", "mil"]}
gap_diff = abs(gaps["mil"] - gaps["mil_t1"])
print(f"판 대조: 다른 판(mil)과의 차이 {gap_diff:.3f} → 허용폭 0.01 밖 = 어긋남")
print("→ mil_t1은 개입 '전' 기저. 무작위배정이라 0 근처 = '효과 없음'으로 오판. 결과변수는 mil.")

### 문제 3 · 표본·결측

**의도**: 패널에서 1차 → 3차로 삶의 의미(mil)가 올랐는지 내렸는지 본다.

AI가 준 코드:

```python
change = panel[panel.wave==3].mil.mean() - panel[panel.wave==1].mil.mean()
```

**명세**: 같은 사람의 변화 = 세 시점에 다 답한 사람만. **잡는 검사**: 인원 검사 = 두 평균이 각각 몇 명의 평균인가.

검사가 실패하면 코드에서 볼 곳: 3차에 남은 사람과 1차 전원은 같은 사람들인가? 중간에 떠난 사람은 누구였나?

In [ ]:
panel = pd.read_csv("data/journey_panel.csv")
w = panel.pivot(index="id", columns="wave", values="mil")
buggy = panel[panel.wave==3].mil.mean() - panel[panel.wave==1].mil.mean()   # 생존자 3차 - 전원 1차
bal = w.dropna()                                                            # 3회 다 참여한 사람만
right = (bal[3] - bal[1]).mean()                                            # 같은 사람의 변화
print(f"버그(생존자 3차 - 전원 1차): {buggy:+.3f}  →  '의미가 올랐다'")
print(f"바름(같은 사람 3차 - 1차)  : {right:+.3f}  →  '의미가 내렸다'")
w1 = panel[panel.wave==1]; stay = set(bal.index)
print(f"이탈자 1차 mil {w1[~w1.id.isin(stay)].mil.mean():.2f} < 잔류자 {w1[w1.id.isin(stay)].mil.mean():.2f}")
n1, n3 = (panel.wave==1).sum(), (panel.wave==3).sum()
print(f"인원 검사: 두 평균의 인원 = 1차 {n1}명 · 3차 {n3}명 (명세 = 같은 사람 {len(bal)}명) → 어긋남")
print("→ 부호가 뒤집힌다. 낮았던 사람이 먼저 떠나(생존 편향), 남은 평균만 올라 보인다. 같은 사람을 따라가야.")

### 문제 4 · 경계

**의도**: 나이 중앙값으로 젊은/나이 든 두 집단을 나눈다(둘을 합치면 전체).

AI가 준 코드:

```python
young = d[d.age <  med]
old   = d[d.age >  med]
```

**명세**: 두 집단의 합 = 전체 인원. **잡는 검사**: 인원 검사 = 젊은 집단 + 나이 든 집단이 전체와 같은가.

검사가 실패하면 코드에서 볼 곳: 나이가 **정확히** 중앙값인 사람은 어느 쪽에 들어가는가?

In [ ]:
d = load("exp"); med = d.age.median()
yb, ob = d[d.age < med], d[d.age > med]         # 버그: < 와 >
yg      = d[d.age <= med]                        # 바름: <=
print(f"버그: 젊은 {len(yb)} + 나이 {len(ob)} = {len(yb)+len(ob)}   (전체 {len(d)})")
print(f"→ {len(d)-len(yb)-len(ob)}명 증발. 나이가 정확히 중앙값({med})인 사람들이 양쪽에서 빠졌다. < 하나를 <= 로.")

### 문제 5 · 방향

**의도**: 개입 집단이 통제보다 의미가 **높은지** 본다(개입 - 통제).

AI가 준 코드:

```python
gap = cg.mean() - tg.mean()   # 통제 - 개입
print(f"차이 {gap:+.3f}")     # 부호를 그대로 문장에 옮기면?
```

**명세**: 차이 = 개입 − 통제 (개입이 높으면 양수). **잡는 검사**: 극단값 시험 = 개입 집단 전원에 1을 더하면 차이가 커져야 한다.

검사가 실패하면 코드에서 볼 곳: 빼는 순서가 의도와 같은가? 부호가 뒤집히면 문장이 어떻게 바뀌나?

In [ ]:
exp = load("exp")
tg, cg = exp[exp.cond==1].mil, exp[exp.cond==0].mil
print(f"버그(통제-개입): {cg.mean()-tg.mean():+.3f}  →  '통제가 더 높다'는 정반대 결론")
print(f"바름(개입-통제): {tg.mean()-cg.mean():+.3f}")
tg_plus = tg + 1                                  # 극단값 시험: 개입 전원 +1
print(f"극단값 시험: 개입 +1 뒤 버그 코드의 차이 {cg.mean()-tg_plus.mean():+.3f} (명세대로라면 {tg_plus.mean()-cg.mean():+.3f}) → 방향 반대")
print("→ t검정 양측 p는 같아도 방향이 뒤집혀 해석이 반대가 된다. 극단값 시험이 그 뒤집힘을 잡는다.")

### 문제 6 · 기본값

**의도**: 삶의 의미(mil)의 표본 표준편차를 구한다.

AI가 준 코드(넘파이로 짬):

```python
sd = np.std(x)     # 기본값은?
```

**명세**: 표본 표준편차(ddof=1). **잡는 검사**: 판 대조(언어 축) = R의 `sd()`로 짠 판과 값이 맞는가.

검사가 실패하면 코드에서 볼 곳: `np.std` 의 기본 ddof는 0(모표준편차)인가 1(표본)인가? pandas와 같은가?

In [ ]:
x = load("svy").mil.values
print(f"버그 np.std 기본(ddof=0): {np.std(x):.4f}")
print(f"바름 표본 SD(ddof=1):     {np.std(x, ddof=1):.4f}   (= pandas .std() 기본값)")
print(f"판 대조(언어 축): R sd() · pandas .std() 판 = {pd.Series(x).std():.4f} → np.std 기본값 판과 어긋남")
print("→ numpy와 pandas의 기본이 다르다. 표본 통계에는 ddof=1. 언어가 다른 판과 맞대면 드러난다.")

## 정리: 명세가 먼저, 코드는 검사가 실패할 때
여섯 오류의 공통점 — **전부 멀쩡히 돌아갔다.** crash 도, 빨간 글씨도 없이 그럴듯한 숫자를 냈다. 수 하나만 봐서는 어느 것이 버그인지 알 수 없었다. 그런데 여섯 모두 코드를 읽지 않고 잡혔다. 결과보다 먼저 정한 명세(기대 인원 · 방향 · 결과변수)에 결과를 걸거나, 다른 판과 맞대면 어긋남이 드러났다. 그래서 AI에게 시키기 전에 명세를 정하고, 받은 결과에 검사(인원 · 판 대조 · 극단값 · 모의 세계)를 건다. 코드는 검사가 실패했을 때, 원인을 찾으려고 연다.

> **연습**: 실제 분석에서 AI에게 일을 시키기 전에, 그 분석의 명세(파일 · 열 · 집단 · 필터 · 기대 인원)를 한 줄씩 써 보라. 받은 결과가 그 명세와 맞는지 검사하는 코드도 AI에게 따로 시킬 수 있다(부록 B §B.3a).